# Example 8.6 / 1: See the Model the Solver Received

Companion notebook for **8.6 How MP Solvers Reformulate Your Model** in the AMPL lab Documentation, section "See the Model the Solver Received". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## See the Model the Solver Received

In this version of the steel model, coils are made in lots of at least 1000 tons or not at all:
`Make['coils'] = 0 or Make['coils'] >= 1000`. HiGHS has no `or`, so its MP driver converts the
disjunction. The driver option `tech:outlev_mp=1` prints the model before and after conversion.
In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
# coils come in lots of at least 1000 tons, or not at all
subject to Coils_Lot: Make['coils'] = 0 or Make['coils'] >= 1000;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", highs_options="tech:outlev_mp=1")
print(ampl.solve_result, ampl.var["Make"].to_dict())
print("AMPL sent", ampl.get_value("_snvars"), "variables and", ampl.get_value("_sncons"), "constraints")

HiGHS 1.14.0:   tech:outlev_mp = 1

AMPL MP initial flat model has 3 variables (0 integer, 0 binary);
Objectives: 1 linear; 
Constraints:  1 linear;
Logical expressions:  2 conditional (in)equalitie(s); 1 or;

AMPL MP final model has 6 variables (0 integer, 2 binary);
Objectives: 1 linear; 
Constraints:  4 linear;


HiGHS 1.14.0: optimal solution; objective 196400
2 simplex iterations
1 branching nodes
solved {'bands': 6000, 'coils': 0, 'plate': 1600}
AMPL sent 3 variables and 1 constraints


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option highs_options 'tech:outlev_mp=1';
solve;
display Make, _snvars, _sncons;

HiGHS 1.14.0:   tech:outlev_mp = 1

AMPL MP initial flat model has 3 variables (0 integer, 0 binary);
Objectives: 1 linear; 
Constraints:  1 linear;
Logical expressions:  2 conditional (in)equalitie(s); 1 or;

AMPL MP final model has 6 variables (0 integer, 2 binary);
Objectives: 1 linear; 
Constraints:  4 linear;


HiGHS 1.14.0: optimal solution; objective 196400
2 simplex iterations
1 branching nodes
Make [*] :=
bands  6000
coils     0
plate  1600
;

_snvars = 3
_sncons = 1

